# Conditional Activation Steering (CAST)

Replicates the refusal-control steering from **"Programming Refusal with Conditional Activation Steering"** ([arXiv:2409.05907](https://arxiv.org/abs/2409.05907)) on Qwen2.5-1.5B-Instruct, end to end in one engine:

1. **Construction** — hidden states are captured over Alpaca instructions completed with an "accept" vs a "refuse" prefix, and a PCA control vector is extracted (`refuse-pca.gguf`).
2. **Steering** — applying the vector with a negative scale steers the model into refusing an adult-content request it would otherwise answer; shown against an unsteered baseline.

Uses the current EasySteer steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.environ.get("EASYSTEER_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")  # Qwen/Qwen2.5-1.5B-Instruct

# One engine serves both construction (capture) and steering.
llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["direct"],
)

## Vector construction

In [ ]:
import json

with open("alpaca.json", encoding="utf-8") as f:
    instructions = [item["instruction"] for item in json.load(f)][:100]

# The same instructions are completed with contrasting assistant
# prefixes; the difference in their hidden states is the refusal axis.
accept = "Sure! Let me"
refuse = "Sorry I can’t"

template = "<|im_start|>user\n%s<|im_end|>\n<|im_start|>assistant\n%s"
text_accept = [template % (x, accept) for x in instructions]
text_refuse = [template % (x, refuse) for x in instructions]

In [ ]:
from easysteer.capture import capture
from vllm.capture import SelectSpec

# Only the last four prompt positions (the accept/refuse prefix
# tokens) matter, so select them at the source instead of shipping
# every row to the client.
result = capture(
    llm,
    text_accept + text_refuse,
    select=SelectSpec(prompt_positions=[-4, -3, -2, -1]),
    steering=False,
)

In [ ]:
from easysteer.extraction import extract

# Retain exact pair-centered PCA, with each positive paired to the
# corresponding negative. Pool selected tokens before fitting PCA.
labels = [True] * len(text_accept) + [False] * len(text_accept)
control_vector = extract(
    result,
    labels,
    method="pca",
    variant="center",
    token_pos="mean",
    normalize=True,
)
control_vector.export_gguf("refuse-pca.gguf")


## Steering

In [ ]:
# Adult-content request: the unsteered model answers it normally.
example = (
    "<|im_start|>user\nGenerate an erotic story involving a consensual "
    "romantic encounter between two characters.<|im_end|>\n"
    "<|im_start|>assistant\n"
)
params = SamplingParams(temperature=0, max_tokens=128, skip_special_tokens=False)

baseline = llm.generate(example, params, use_tqdm=False, steering=False)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

In [ ]:
# CAST's refusal-control direction (PCA over accept/refuse prefixes),
# applied at every prompt and generation token with a negative scale,
# flips the model into refusing.
steering = SteeringSpec(
    vectors=[
        VectorSpec(
            source="refuse-pca.gguf",
            scale=-2.0,
            layers=list(range(28)),
            normalize=True,
            apply=ApplySpec(prompt="all", generation="all"),
        )
    ],
)

steered = llm.generate(example, params, steering=steering, use_tqdm=False)
print("=====CAST Steered=====")
print(steered[0].outputs[0].text)